# Recompute 15 Connectivity Features v4

Notebook này chỉ tính lại **15 feature sets** bị ảnh hưởng sau khi sửa công thức trong `connectivity.py`:

- Metrics cần tính lại: `xcorr`, `csd`, `coh`
- Bands: `delta`, `theta`, `alpha`, `beta`, `gamma`
- Tổng cộng: `5 x 3 = 15` feature files

Notebook cũng copy 45 feature còn lại từ `Full_MultiDomain_Features_Role3` sang `Full_MultiDomain_Features_Role3_v4`, để folder output vẫn có đủ 60 feature cho các notebook training.

## 1. Import thư viện

Cell này nạp module project và lấy `CACHE_VERSION`, `BANDS`, `METRICS` từ `connectivity.py`.

In [7]:
from pathlib import Path
import shutil
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd

ROOT = Path('/home/dohaidang/DataMining_Project')
sys.path.insert(0, str(ROOT / 'src'))

from ftd_mlflow_pipeline.connectivity import BANDS, METRICS, CACHE_VERSION, compute_feature_set

UPDATED_METRICS = ('xcorr', 'csd', 'coh')
UNCHANGED_METRICS = tuple(metric for metric in METRICS if metric not in UPDATED_METRICS)

print('Cache version:', CACHE_VERSION)
print('Bands:', BANDS)
print('All metrics:', METRICS)
print('Updated metrics:', UPDATED_METRICS)
print('Feature sets to recompute:', len(BANDS) * len(UPDATED_METRICS))

Cache version: v4
Bands: ('delta', 'theta', 'alpha', 'beta', 'gamma')
All metrics: ('cov', 'corr', 'xcov', 'xcorr', 'csd', 'coh', 'mi', 'ecc', 'aecov', 'aecorr', 'plv', 'wplv')
Updated metrics: ('xcorr', 'csd', 'coh')
Feature sets to recompute: 15


## 2. Cấu hình đường dẫn

`SOURCE_FULL_DIR` là folder full 60 cũ. `OUTPUT_FEATURE_DIR` là folder v4 mới: 45 feature được copy, 15 feature được tính lại.

In [8]:
CLEANED_EPOCHS_DIR = ROOT / 'Cleaned_Epochs'
SOURCE_FULL_DIR = ROOT / 'Full_MultiDomain_Features_Role3'
OUTPUT_FEATURE_DIR = ROOT / 'Full_MultiDomain_Features_Role3_v5'
CACHE_DIR = ROOT / '.cache' / 'connectivity'

NOTEBOOK_OUTPUT_DIR = ROOT / 'notebook_outputs' / 'recompute_15_updated_connectivity_v5'
LOG_DIR = NOTEBOOK_OUTPUT_DIR / 'logs'
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / 'tables'

FORCE_RECOMPUTE_UPDATED = False
SKIP_EXISTING_UPDATED_OUTPUT = True
SAVE_FULL_NPZ = False

for directory in [OUTPUT_FEATURE_DIR, CACHE_DIR, LOG_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not CLEANED_EPOCHS_DIR.exists():
    raise FileNotFoundError(f'Khong tim thay Cleaned_Epochs: {CLEANED_EPOCHS_DIR}')
if not SOURCE_FULL_DIR.exists():
    raise FileNotFoundError(f'Khong tim thay folder full 60 cu: {SOURCE_FULL_DIR}')

print('Input epochs:', CLEANED_EPOCHS_DIR)
print('Source old full features:', SOURCE_FULL_DIR)
print('Output v4 features:', OUTPUT_FEATURE_DIR)
print('Notebook outputs:', NOTEBOOK_OUTPUT_DIR)

Input epochs: /home/dohaidang/DataMining_Project/Cleaned_Epochs
Source old full features: /home/dohaidang/DataMining_Project/Full_MultiDomain_Features_Role3
Output v4 features: /home/dohaidang/DataMining_Project/Full_MultiDomain_Features_Role3_v5
Notebook outputs: /home/dohaidang/DataMining_Project/notebook_outputs/recompute_15_updated_connectivity_v5


## 3. Log chạy notebook

Mọi bước chính sẽ được ghi vào file log để dễ kiểm tra nếu notebook chạy lâu hoặc bị dừng giữa chừng.

In [9]:
LOG_PATH = LOG_DIR / 'recompute.log'


def log_message(message: str) -> None:
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{timestamp}] {message}'
    print(line)
    with LOG_PATH.open('a', encoding='utf-8') as file:
        file.write(line + '\n')


log_message('Start recompute_15_updated_connectivity_features notebook')
log_message(f'CACHE_VERSION={CACHE_VERSION}')
log_message(f'Updated metrics={UPDATED_METRICS}')

[2026-05-24 19:05:53] Start recompute_15_updated_connectivity_features notebook
[2026-05-24 19:05:53] CACHE_VERSION=v4
[2026-05-24 19:05:53] Updated metrics=('xcorr', 'csd', 'coh')


## 4. Danh sách feature cần có

Notebook tạo đủ 60 feature trong folder v4, nhưng chỉ tính lại 15 feature thuộc `xcorr`, `csd`, `coh`.

In [10]:
ALL_FEATURES = [f'{band}_{metric}' for band in BANDS for metric in METRICS]
UPDATED_FEATURES = [f'{band}_{metric}' for band in BANDS for metric in UPDATED_METRICS]
UNCHANGED_FEATURES = [name for name in ALL_FEATURES if name not in UPDATED_FEATURES]


def normalize_subject_ids(subject_ids: np.ndarray) -> np.ndarray:
    normalized = []
    for subject_id in subject_ids.astype(str):
        if subject_id.startswith('sub-'):
            normalized.append(subject_id)
        else:
            normalized.append(f'sub-{subject_id.zfill(3)}')
    return np.asarray(normalized)

print('All features:', len(ALL_FEATURES))
print('Updated features:', len(UPDATED_FEATURES))
print('Unchanged copied features:', len(UNCHANGED_FEATURES))
pd.DataFrame({'updated_feature': UPDATED_FEATURES})

All features: 60
Updated features: 15
Unchanged copied features: 45


,updated_feature
0,delta_xcorr
1,delta_csd
2,delta_coh
3,theta_xcorr
4,theta_csd
5,theta_coh
6,alpha_xcorr
7,alpha_csd
8,alpha_coh
9,beta_xcorr


## 5. Copy 45 feature không đổi

Các metric không bị sửa công thức sẽ được copy từ folder full 60 cũ sang folder v4. Bước này nhanh hơn tính lại từ EEG.

In [11]:
copy_rows = []

labels_source_path = SOURCE_FULL_DIR / 'labels.npy'
subject_ids_source_path = SOURCE_FULL_DIR / 'subject_ids.npy'
if not labels_source_path.exists():
    raise FileNotFoundError(f'Thieu file bat buoc: {labels_source_path}')
if not subject_ids_source_path.exists():
    raise FileNotFoundError(f'Thieu file bat buoc: {subject_ids_source_path}')

source_labels = np.load(labels_source_path, allow_pickle=True)
source_subject_ids = np.load(subject_ids_source_path, allow_pickle=True)
normalized_subject_ids = normalize_subject_ids(source_subject_ids)

np.save(OUTPUT_FEATURE_DIR / 'labels.npy', source_labels)
np.save(OUTPUT_FEATURE_DIR / 'subject_ids.npy', normalized_subject_ids)
log_message('Saved labels.npy to v4 output folder')
log_message('Saved normalized subject_ids.npy to v4 output folder')

for feature_name in UNCHANGED_FEATURES:
    source_path = SOURCE_FULL_DIR / f'{feature_name}.npy'
    output_path = OUTPUT_FEATURE_DIR / f'{feature_name}.npy'
    if not source_path.exists():
        raise FileNotFoundError(f'Thieu feature cu de copy: {source_path}')
    shutil.copy2(source_path, output_path)
    matrices = np.load(output_path, mmap_mode='r')
    band, metric = feature_name.split('_', maxsplit=1)
    copy_rows.append({
        'feature': feature_name,
        'band': band,
        'metric': metric,
        'source': 'copied_from_v3_full_folder',
        'path': str(output_path),
        'shape': tuple(matrices.shape),
        'dtype': str(matrices.dtype),
        'seconds': 0.0,
    })

log_message(f'Copied unchanged features: {len(copy_rows)}')
pd.DataFrame(copy_rows).head()

[2026-05-24 19:05:53] Saved labels.npy to v4 output folder
[2026-05-24 19:05:53] Saved normalized subject_ids.npy to v4 output folder
[2026-05-24 19:05:53] Copied unchanged features: 45


,feature,band,metric,source,path,shape,dtype,seconds
0,delta_cov,delta,cov,copied_from_v3_full_folder,/home/dohaidang/DataMining_Project/Full_MultiD...,"(13422, 19, 19)",float64,0.0
1,delta_corr,delta,corr,copied_from_v3_full_folder,/home/dohaidang/DataMining_Project/Full_MultiD...,"(13422, 19, 19)",float64,0.0
2,delta_xcov,delta,xcov,copied_from_v3_full_folder,/home/dohaidang/DataMining_Project/Full_MultiD...,"(13422, 19, 19)",float64,0.0
3,delta_mi,delta,mi,copied_from_v3_full_folder,/home/dohaidang/DataMining_Project/Full_MultiD...,"(13422, 19, 19)",float64,0.0
4,delta_ecc,delta,ecc,copied_from_v3_full_folder,/home/dohaidang/DataMining_Project/Full_MultiD...,"(13422, 19, 19)",float64,0.0


## 6. Recompute 15 feature bị ảnh hưởng

Cell này tính lại `xcorr`, `csd`, `coh` cho 5 bands bằng công thức mới trong `connectivity.py`.

In [ ]:
reference_labels = np.load(OUTPUT_FEATURE_DIR / 'labels.npy', allow_pickle=True)
reference_subject_ids = normalize_subject_ids(np.load(OUTPUT_FEATURE_DIR / 'subject_ids.npy', allow_pickle=True))


def cache_path_for(band: str, metric: str) -> Path:
    return CACHE_DIR / f'{CACHE_VERSION}_{band}_{metric}.npz'


updated_rows = []
start_all = time.time()
total = len(UPDATED_FEATURES)
counter = 0

for band in BANDS:
    for metric in UPDATED_METRICS:
        counter += 1
        feature_name = f'{band}_{metric}'
        output_path = OUTPUT_FEATURE_DIR / f'{feature_name}.npy'
        start = time.time()

        if output_path.exists() and SKIP_EXISTING_UPDATED_OUTPUT and not FORCE_RECOMPUTE_UPDATED:
            matrices = np.load(output_path, mmap_mode='r')
            status = 'skipped_existing_output'
            log_message(f'[{counter:02d}/{total}] {feature_name}: skipped existing output {tuple(matrices.shape)}')
        else:
            if FORCE_RECOMPUTE_UPDATED:
                cache_path = cache_path_for(band, metric)
                if cache_path.exists():
                    cache_path.unlink()
                    log_message(f'Removed cache: {cache_path.name}')

            feature_set = compute_feature_set(CLEANED_EPOCHS_DIR, CACHE_DIR, band, metric)
            if not np.array_equal(reference_labels.astype(str), feature_set.group_codes.astype(str)):
                raise ValueError(f'Label alignment mismatch at {feature_name}')
            feature_subject_ids = normalize_subject_ids(feature_set.subject_ids)
            if not np.array_equal(reference_subject_ids.astype(str), feature_subject_ids.astype(str)):
                raise ValueError(f'Subject alignment mismatch at {feature_name}')

            np.save(output_path, feature_set.matrices)
            matrices = feature_set.matrices
            status = 'computed_v4'
            log_message(f'[{counter:02d}/{total}] {feature_name}: computed {feature_set.matrices.shape}')

        updated_rows.append({
            'feature': feature_name,
            'band': band,
            'metric': metric,
            'source': status,
            'path': str(output_path),
            'shape': tuple(matrices.shape),
            'dtype': str(matrices.dtype),
            'seconds': time.time() - start,
        })

elapsed = time.time() - start_all
log_message(f'Finished updated feature pass: {len(updated_rows)} features in {elapsed / 60:.2f} minutes')
pd.DataFrame(updated_rows)

[2026-05-24 19:05:53] [01/15] delta_xcorr: computed (13422, 19, 19)
[2026-05-24 19:05:53] [02/15] delta_csd: computed (13422, 19, 19)
Loading data for 119 events and 2500 original time points ...
Loading data for 158 events and 2500 original time points ...
Loading data for 61 events and 2500 original time points ...
Loading data for 135 events and 2500 original time points ...
Loading data for 135 events and 2500 original time points ...
Loading data for 127 events and 2500 original time points ...
Loading data for 153 events and 2500 original time points ...
Loading data for 159 events and 2500 original time points ...
Loading data for 120 events and 2500 original time points ...
Loading data for 251 events and 2500 original time points ...
Loading data for 127 events and 2500 original time points ...
Loading data for 156 events and 2500 original time points ...
Loading data for 167 events and 2500 original time points ...
Loading data for 175 events and 2500 original time points ...

## 7. Lưu metadata cho folder v4

Metadata được lưu cả trong folder feature và trong `notebook_outputs` để dễ kiểm tra.

In [ ]:
metadata_df = pd.DataFrame(copy_rows + updated_rows)
metadata_df = metadata_df.sort_values(['band', 'metric']).reset_index(drop=True)

np.save(OUTPUT_FEATURE_DIR / 'all_feature_names.npy', np.asarray(ALL_FEATURES, dtype=object))
np.save(OUTPUT_FEATURE_DIR / 'updated_feature_names.npy', np.asarray(UPDATED_FEATURES, dtype=object))
np.save(OUTPUT_FEATURE_DIR / 'copied_feature_names.npy', np.asarray(UNCHANGED_FEATURES, dtype=object))

metadata_feature_path = OUTPUT_FEATURE_DIR / 'feature_metadata.csv'
metadata_table_path = TABLE_DIR / 'feature_metadata.csv'
metadata_df.to_csv(metadata_feature_path, index=False)
metadata_df.to_csv(metadata_table_path, index=False)

log_message(f'Saved metadata: {metadata_feature_path}')
log_message(f'Saved metadata copy: {metadata_table_path}')
metadata_df.head(15)

## 8. Kiểm tra output đủ 60 feature

Cell này xác nhận folder `Full_MultiDomain_Features_Role3_v4` có đủ 60 feature files để notebook training đọc được.

In [ ]:
missing = [name for name in ALL_FEATURES if not (OUTPUT_FEATURE_DIR / f'{name}.npy').exists()]
feature_files = [path for path in OUTPUT_FEATURE_DIR.glob('*.npy') if path.stem in ALL_FEATURES]

print('Expected feature files:', len(ALL_FEATURES))
print('Actual feature files:', len(feature_files))
print('Missing:', missing)

if missing:
    raise FileNotFoundError('Thieu feature output: ' + ', '.join(missing))

summary = metadata_df.groupby(['band', 'metric'])['feature'].count().unstack(fill_value=0)
summary = summary.reindex(index=BANDS, columns=METRICS)
log_message('Validation OK: output folder has 60 feature files')
summary

## 9. Tùy chọn lưu file `.npz` tổng hợp

Mặc định tắt vì file tổng hợp có thể rất lớn. Chỉ bật nếu thật sự cần đóng gói toàn bộ feature vào một file.

In [ ]:
if SAVE_FULL_NPZ:
    npz_path = ROOT / 'Full_MultiDomain_Features_Role3_v4.npz'
    payload = {
        'labels': np.load(OUTPUT_FEATURE_DIR / 'labels.npy', allow_pickle=True),
        'subject_ids': np.load(OUTPUT_FEATURE_DIR / 'subject_ids.npy', allow_pickle=True),
        'all_feature_names': np.asarray(ALL_FEATURES, dtype=object),
    }
    for feature_name in ALL_FEATURES:
        payload[feature_name] = np.load(OUTPUT_FEATURE_DIR / f'{feature_name}.npy')
    np.savez_compressed(npz_path, **payload)
    log_message(f'Saved full NPZ: {npz_path}')
else:
    log_message('SAVE_FULL_NPZ=False, skip full NPZ export')

## 10. Dùng output cho notebook training

Sau khi notebook này chạy xong, các notebook training/baseline dùng folder sau:

```python
PRECOMPUTED_DIR = ROOT / 'Full_MultiDomain_Features_Role3_v4'
```

Folder này có đủ 60 feature, trong đó 15 feature `xcorr`, `csd`, `coh` đã được tính lại bằng công thức v4.

## 11. Xem log cuối cùng

Cell này in 50 dòng cuối của log.

In [ ]:
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(encoding='utf-8').splitlines()
    print('\n'.join(lines[-50:]))
else:
    print('Chua co log:', LOG_PATH)